# Old API vs New API Demo

In [6]:
import warnings

import torch

from sbi.inference import NLE, MCMCPosterior
from sbi.utils import BoxUniform

warnings.filterwarnings('default')


# Setup
prior = BoxUniform(low=torch.zeros(2), high=torch.ones(2))
def simulator(θ):
    return θ + torch.randn_like(θ) * 0.1

torch.manual_seed(42)
theta_train = prior.sample((1000,))
x_train = simulator(theta_train)
x_o = torch.randn(1, 2)

# Train
trainer = NLE()
likelihood_estimator = trainer.append_simulations(theta_train, x_train).train()

 Neural network successfully converged after 77 epochs.

## OLD API (Deprecated)

The old API uses stateful classes that bind `x_o` at construction time.

In [7]:
# OLD API: stateful, emits DeprecationWarning
from sbi.inference.potentials import likelihood_estimator_based_potential

potential_fn, parameter_transform = likelihood_estimator_based_potential(
    likelihood_estimator, prior, x_o
)
posterior_old = MCMCPosterior(
    potential_fn, proposal=prior, theta_transform=parameter_transform, warmup_steps=10
)
samples_old = posterior_old.sample((1000,))
print(f"OLD API - samples mean: {samples_old.mean(dim=0)}")

Running vectorized MCMC with 20 chains: 100%|██████████| 2200/2200 [00:03<00:00, 594.07it/s] 

OLD API - samples mean: tensor([0.9753, 0.0945])


## NEW API (Recommended)

The new API uses stateless functions that follow the `PotentialFunction` protocol:
- `(theta, x) -> log_prob`
- No hidden state, all inputs explicit
- Combine with `bind_observation()`

In [8]:
# NEW API: stateless + bind_observation
from sbi.inference.potentials import bind_observation, likelihood_potential
from sbi.utils import mcmc_transform

# Step 1: Create stateless potential (satisfies PotentialFunction protocol)
potential_fn = likelihood_potential(likelihood_estimator, prior)

# Step 2: Get transform from prior (for MCMC in unconstrained space)
theta_transform = mcmc_transform(prior, enable_transform=True)

# Step 3: Bind observation to create sampler-ready function
bound_fn = bind_observation(potential_fn, x_o, sum_iid=True)

# Step 4: Use with MCMC sampler (pass theta_transform!)
posterior_new = MCMCPosterior(
    bound_fn, proposal=prior, theta_transform=theta_transform, warmup_steps=10
)
samples_new = posterior_new.sample((1000,))
print(f"NEW API - samples mean: {samples_new.mean(dim=0)}")

Running vectorized MCMC with 20 chains: 100%|██████████| 2200/2200 [00:05<00:00, 425.35it/s]

NEW API - samples mean: tensor([0.9765, 0.0895])


## Lower-Level: Using the Protocol Directly

You can also implement your own potential function that satisfies the protocol.

In [9]:
# Custom potential function satisfying PotentialFunction protocol
def my_custom_potential(theta: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Custom potential: log p(x|theta) + log p(theta)"""
    log_likelihood = likelihood_estimator.log_prob(x, condition=theta)
    log_prior = prior.log_prob(theta)
    return log_likelihood + log_prior

# Add device attribute for bind_observation compatibility
my_custom_potential.device = next(likelihood_estimator.parameters()).device

# Bind observation
bound_custom = bind_observation(my_custom_potential, x_o, sum_iid=True)

# Use with sampler
posterior_custom = MCMCPosterior(
    bound_custom, proposal=prior, warmup_steps=10
)
samples_custom = posterior_custom.sample((1000,))
print(f"Custom potential - samples mean: {samples_custom.mean(dim=0)}")

Running vectorized MCMC with 20 chains: 100%|██████████| 2200/2200 [00:02<00:00, 998.14it/s] 

Custom potential - samples mean: tensor([0.9757, 0.0981])


In [10]:
print(f"OLD API:  {samples_old.mean(dim=0)}")
print(f"NEW API:  {samples_new.mean(dim=0)}")
print(f"Custom:   {samples_custom.mean(dim=0)}")

OLD API:  tensor([0.9753, 0.0945])
NEW API:  tensor([0.9765, 0.0895])
Custom:   tensor([0.9757, 0.0981])
